# 15. Diplomatic subindex integration

Integrate diplomatic structural ties and United Nations General Assembly voting alignment into a signed country-year subindex. Positive values indicate greater closeness to the United States; negative values indicate greater closeness to China.

This notebook is a development and validation record. It does not write production outputs.

## Baseline period and latest-year policy

The complete-data baseline period for the current index edition is **2004–2024**. The year **2025** is retained only as a clearly flagged provisional extension because the diplomatic-representation source currently ends in 2024.

When both components are available, use equal conceptual weights. When only one component is available, renormalise to that component. When neither is available, retain a missing score. Available-component renormalisation never makes a one-component observation compositionally comparable with the two-component baseline.

The broad UNGA specification is the baseline. The strict Yes-versus-No score, alternative component weights, zero filling, and exclusion of provisionally insufficient UNGA participation are sensitivities.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DIPLOMATIC_PATH = ROOT / "data/processed/diplomatic_structural_ties_country_anchor_year.csv"
UNGA_PATH = ROOT / "data/processed/unga_country_year_scores.csv"
COUNTRY_MASTER_PATH = ROOT / "outputs/tables/relations_condition/relationship_condition_country_master.csv"

diplomatic_raw = pd.read_csv(DIPLOMATIC_PATH, low_memory=False)
unga_raw = pd.read_csv(UNGA_PATH, low_memory=False)
country_master = pd.read_csv(COUNTRY_MASTER_PATH, low_memory=False)

pd.Series({
    "diplomatic_rows": len(diplomatic_raw),
    "unga_rows": len(unga_raw),
    "country_master_rows": len(country_master),
}, name="value")

diplomatic_rows        8410
unga_rows              4392
country_master_rows     213
Name: value, dtype: int64

## 1. Validate and reshape diplomatic structural ties

The source ends in 2024, so 2025 structural scores are expected to be missing for both anchors.

In [2]:
required = {"country_iso3", "preferred_country_name", "anchor", "year", "diplomatic_structural_score_raw", "source_coverage_status"}
missing = required - set(diplomatic_raw.columns)
if missing:
    raise ValueError(f"Diplomatic panel missing columns: {sorted(missing)}")
if diplomatic_raw.duplicated(["country_iso3", "anchor", "year"]).any():
    raise ValueError("Duplicate diplomatic country-anchor-year keys detected.")
if not diplomatic_raw["diplomatic_structural_score_raw"].dropna().between(0, 1).all():
    raise ValueError("Diplomatic structural scores fall outside [0, 1].")

structural = diplomatic_raw.pivot(
    index=["country_iso3", "preferred_country_name", "year"],
    columns="anchor",
    values="diplomatic_structural_score_raw",
).reset_index()
if not {"China", "United States"}.issubset(structural.columns):
    raise ValueError("Diplomatic panel must contain China and United States anchors.")
structural["diplomatic_structural_signed_score"] = structural["United States"] - structural["China"]
if not structural["diplomatic_structural_signed_score"].dropna().between(-1, 1).all():
    raise ValueError("Signed structural scores fall outside [-1, 1].")

pd.Series({
    "rows": len(structural),
    "entities": structural["country_iso3"].nunique(),
    "first_year": structural["year"].min(),
    "last_year": structural["year"].max(),
    "duplicate_country_years": structural.duplicated(["country_iso3", "year"]).sum(),
    "missing_signed_scores": structural["diplomatic_structural_signed_score"].isna().sum(),
}, name="value")

rows                       4205
entities                    192
first_year                 2004
last_year                  2025
duplicate_country_years       0
missing_signed_scores       191
Name: value, dtype: int64

## 2. Map and validate UNGA scores

Use the canonical country master and three explicit source-name aliases. Retain provisionally insufficient participation cases because they have calculated scores; preserve their status and participation rate as diagnostics.

In [3]:
eligible_master = country_master.loc[
    country_master["baseline_index_eligible"].eq(True)
].copy()

name_to_iso3 = {}
for row in eligible_master.itertuples(index=False):
    for name in (row.country_name, row.preferred_country_name):
        if pd.notna(name):
            name_to_iso3[str(name)] = row.iso3

UNGA_NAME_ALIASES = {
    "Federated States of Micronesia": "FSM",
    "Gambia": "GMB",
    "Saint Vincent and the Grenadines": "VCT",
    "Côte d'Ivoire": "CIV",
    "São Tomé and Príncipe": "STP",
    "Türkiye": "TUR",
}
name_to_iso3.update(UNGA_NAME_ALIASES)

unga = unga_raw.loc[unga_raw["year"].between(2004, 2025)].copy()
unga["iso3"] = unga["country_name"].map(name_to_iso3)

unmapped = sorted(
    unga.loc[unga["iso3"].isna(), "country_name"].dropna().unique()
)
if unmapped:
    raise ValueError(f"Unmapped UNGA countries: {unmapped}")
if unga.duplicated(["iso3", "year"]).any():
    raise ValueError("Duplicate UNGA ISO3-year keys detected.")
if not unga["unga_score_broad"].dropna().between(-1, 1).all():
    raise ValueError("Broad UNGA scores fall outside [-1, 1].")

valid_statuses = {
    "available",
    "provisionally_insufficient_participation",
    "no_country_participation",
}
unexpected = sorted(set(unga["score_status"].dropna()) - valid_statuses)
if unexpected:
    raise ValueError(f"Unexpected UNGA statuses: {unexpected}")

pd.DataFrame({
    "rows": unga.groupby("score_status").size(),
    "missing_broad_score": unga.groupby("score_status")["unga_score_broad"].apply(
        lambda values: values.isna().sum()
    ),
    "mean_participation_rate": unga.groupby("score_status")[
        "broad_participation_rate"
    ].mean(),
})

,rows,missing_broad_score,mean_participation_rate
score_status,,,
available,4037,0,0.944019
no_country_participation,15,15,0.000000
provisionally_insufficient_participation,149,0,0.274962


## 3. Join the components and construct the baseline

The diplomatic structural panel defines the project universe. UNGA rows must not fall outside that universe; eligible diplomatic country-years may lack a matching UNGA observation and must remain explicitly represented.

Preserve separate structural-availability, UNGA-availability, integration, and comparability fields so that country-specific missingness is not conflated with whole-year source unavailability.

In [4]:
unga_keep = unga[[
    "iso3", "year", "unga_score_broad", "unga_score_strict_yn",
    "score_status", "broad_participation_rate",
    "broad_eligible_resolutions", "broad_participated_votes",
]].copy()

integrated = structural.merge(
    unga_keep,
    left_on=["country_iso3", "year"],
    right_on=["iso3", "year"],
    how="outer",
    validate="one_to_one",
    indicator="join_status",
)

join_counts = integrated["join_status"].value_counts().reindex(
    ["both", "left_only", "right_only"], fill_value=0
)
if join_counts["right_only"] != 0:
    unexpected_rows = integrated.loc[
        integrated["join_status"].eq("right_only"), ["iso3", "year"]
    ].sort_values(["iso3", "year"])
    raise ValueError(
        "UNGA contains country-years outside the diplomatic production "
        f"universe: {unexpected_rows.to_dict('records')}"
    )

expected_left_only = {
    ("SCG", 2004), ("SCG", 2005), ("SCG", 2006), ("TUR", 2025)
}
actual_left_only = set(
    integrated.loc[
        integrated["join_status"].eq("left_only"),
        ["country_iso3", "year"],
    ].itertuples(index=False, name=None)
)
if actual_left_only != expected_left_only:
    raise ValueError(
        "Unexpected diplomatic country-years without matching UNGA rows. "
        f"Expected {sorted(expected_left_only)}; found {sorted(actual_left_only)}"
    )

join_audit = integrated.loc[
    integrated["join_status"].ne("both"),
    ["country_iso3", "preferred_country_name", "year", "join_status"],
].sort_values(["country_iso3", "year"])

display(join_counts)
display(join_audit)

integrated = integrated.drop(columns=["iso3"])
structural_available = integrated[
    "diplomatic_structural_signed_score"
].notna()
unga_available = integrated["unga_score_broad"].notna()
both_available = structural_available & unga_available

integrated["structural_availability_status"] = np.where(
    structural_available,
    "available",
    np.where(
        integrated["year"].eq(2025),
        "source_year_unavailable",
        "unavailable_other",
    ),
)
integrated["unga_availability_status"] = np.select(
    [
        unga_available,
        integrated["score_status"].eq("no_country_participation"),
        integrated["join_status"].eq("left_only"),
    ],
    [
        "available",
        "no_country_participation",
        "country_year_row_unavailable",
    ],
    default="unavailable_other",
)

integrated["diplomatic_integration_status"] = np.select(
    [
        both_available,
        structural_available & ~unga_available
        & integrated["score_status"].eq("no_country_participation"),
        structural_available & integrated["join_status"].eq("left_only"),
        ~structural_available & unga_available,
        ~structural_available & ~unga_available
        & integrated["score_status"].eq("no_country_participation"),
        ~structural_available & integrated["join_status"].eq("left_only"),
    ],
    [
        "both_components_available",
        "structural_only_unga_no_participation",
        "structural_only_unga_row_unavailable",
        "unga_only_structural_source_year_unavailable",
        "neither_unga_no_participation_and_structural_unavailable",
        "neither_unga_row_missing_and_structural_unavailable",
    ],
    default="invalid",
)

integrated["diplomatic_score_baseline"] = np.select(
    [both_available, structural_available, unga_available],
    [
        0.5 * integrated["diplomatic_structural_signed_score"]
        + 0.5 * integrated["unga_score_broad"],
        integrated["diplomatic_structural_signed_score"],
        integrated["unga_score_broad"],
    ],
    default=np.nan,
)

integrated["diplomatic_comparability_status"] = np.select(
    [
        integrated["year"].eq(2025) & integrated["diplomatic_score_baseline"].notna(),
        integrated["year"].eq(2025) & integrated["diplomatic_score_baseline"].isna(),
        integrated["year"].le(2024) & both_available,
        integrated["year"].le(2024) & integrated["diplomatic_score_baseline"].notna(),
    ],
    [
        "provisional_latest_year_extension",
        "score_unavailable",
        "comparable_complete_component_period",
        "available_component_renormalised",
    ],
    default="invalid",
)

if not integrated["diplomatic_score_baseline"].dropna().between(-1, 1).all():
    raise ValueError("Diplomatic baseline scores fall outside [-1, 1].")

status_summary = integrated.groupby(
    ["diplomatic_integration_status", "diplomatic_comparability_status"],
    dropna=False,
).agg(
    country_years=("country_iso3", "size"),
    countries=("country_iso3", "nunique"),
    mean_structural=("diplomatic_structural_signed_score", "mean"),
    mean_unga=("unga_score_broad", "mean"),
    mean_diplomatic=("diplomatic_score_baseline", "mean"),
).reset_index()
status_summary["share"] = status_summary["country_years"] / len(integrated)
status_summary

join_status
both          4201
left_only        4
right_only       0
Name: count, dtype: int64

,country_iso3,preferred_country_name,year,join_status
3256,SCG,Serbia and Montenegro,2004,left_only
3257,SCG,Serbia and Montenegro,2005,left_only
3258,SCG,Serbia and Montenegro,2006,left_only
3874,TUR,Turkiye,2025,left_only


,diplomatic_integration_status,diplomatic_comparability_status,country_years,countries,mean_structural,mean_unga,mean_diplomatic,share
0,both_components_available,comparable_complete_component_period,3998,191,0.020358,-0.529531,-0.254587,0.950773
1,neither_unga_no_participation_and_structural_u...,score_unavailable,2,2,NaN,NaN,NaN,0.000476
2,neither_unga_row_missing_and_structural_unavai...,score_unavailable,1,1,NaN,NaN,NaN,0.000238
3,structural_only_unga_no_participation,available_component_renormalised,13,4,-0.228365,NaN,-0.228365,0.003092
4,structural_only_unga_row_unavailable,available_component_renormalised,3,1,-0.052083,NaN,-0.052083,0.000713
5,unga_only_structural_source_year_unavailable,provisional_latest_year_extension,188,188,NaN,-0.761076,-0.761076,0.044709


## 4. Missingness, weighting, participation, and UNGA-specification sensitivities

Compare available-component renormalisation with zero filling. Evaluate alternative component weights only on common two-component support so that unchanged one-component rows do not dilute the reported effect. Also test the strict UNGA specification and exclusion of provisionally insufficient participation.

In [5]:
integrated["diplomatic_score_zero_fill"] = 0.5 * (
    integrated["diplomatic_structural_signed_score"].fillna(0) + integrated["unga_score_broad"].fillna(0)
)
missingness_change = (integrated["diplomatic_score_baseline"] - integrated["diplomatic_score_zero_fill"]).abs()
affected = missingness_change.gt(1e-12) & integrated["diplomatic_score_baseline"].notna()

missingness_summary = pd.Series({
    "affected_country_years": int(affected.sum()),
    "maximum_absolute_change": missingness_change[affected].max(),
    "mean_absolute_change_among_affected": missingness_change[affected].mean(),
    "country_year_sign_changes": int((np.sign(integrated.loc[affected, "diplomatic_score_baseline"]) != np.sign(integrated.loc[affected, "diplomatic_score_zero_fill"])).sum()),
}, name="value")
missingness_summary

affected_country_years                 193.000000
maximum_absolute_change                  0.500000
mean_absolute_change_among_affected      0.384323
country_year_sign_changes                0.000000
Name: value, dtype: float64

In [6]:
integrated["diplomatic_score_structural_emphasis"] = np.select(
    [both_available, structural_available, unga_available],
    [
        (2/3) * integrated["diplomatic_structural_signed_score"]
        + (1/3) * integrated["unga_score_broad"],
        integrated["diplomatic_structural_signed_score"],
        integrated["unga_score_broad"],
    ],
    default=np.nan,
)
integrated["diplomatic_score_unga_emphasis"] = np.select(
    [both_available, structural_available, unga_available],
    [
        (1/3) * integrated["diplomatic_structural_signed_score"]
        + (2/3) * integrated["unga_score_broad"],
        integrated["diplomatic_structural_signed_score"],
        integrated["unga_score_broad"],
    ],
    default=np.nan,
)

strict_available = integrated["unga_score_strict_yn"].notna()
strict_both = structural_available & strict_available
integrated["diplomatic_score_strict_unga"] = np.select(
    [strict_both, structural_available, strict_available],
    [
        0.5 * integrated["diplomatic_structural_signed_score"]
        + 0.5 * integrated["unga_score_strict_yn"],
        integrated["diplomatic_structural_signed_score"],
        integrated["unga_score_strict_yn"],
    ],
    default=np.nan,
)

participation_acceptable = ~integrated["score_status"].eq(
    "provisionally_insufficient_participation"
)
unga_participation_sensitivity = (
    integrated["unga_score_broad"].where(participation_acceptable)
)
unga_participation_available = unga_participation_sensitivity.notna()
participation_both = structural_available & unga_participation_available
integrated["diplomatic_score_exclude_provisional_participation"] = np.select(
    [participation_both, structural_available, unga_participation_available],
    [
        0.5 * integrated["diplomatic_structural_signed_score"]
        + 0.5 * unga_participation_sensitivity,
        integrated["diplomatic_structural_signed_score"],
        unga_participation_sensitivity,
    ],
    default=np.nan,
)

score_columns = [
    "diplomatic_score_baseline",
    "diplomatic_score_zero_fill",
    "diplomatic_score_structural_emphasis",
    "diplomatic_score_unga_emphasis",
    "diplomatic_score_strict_unga",
    "diplomatic_score_exclude_provisional_participation",
]
display(integrated[score_columns].corr())

sensitivity_specs = {
    "structural_emphasis": (
        "diplomatic_score_structural_emphasis", both_available
    ),
    "unga_emphasis": (
        "diplomatic_score_unga_emphasis", both_available
    ),
    "strict_unga": (
        "diplomatic_score_strict_unga", both_available & strict_available
    ),
    "exclude_provisional_participation": (
        "diplomatic_score_exclude_provisional_participation",
        integrated["score_status"].eq("provisionally_insufficient_participation")
        & integrated["diplomatic_score_baseline"].notna(),
    ),
}

rows = []
for label, (column, target) in sensitivity_specs.items():
    scores_lost = int((
        target
        & integrated["diplomatic_score_baseline"].notna()
        & integrated[column].isna()
    ).sum())
    comparable = (
        target
        & integrated["diplomatic_score_baseline"].notna()
        & integrated[column].notna()
    )
    change = (
        integrated.loc[comparable, column]
        - integrated.loc[comparable, "diplomatic_score_baseline"]
    ).abs()
    rows.append({
        "specification": label,
        "target_country_years": int(target.sum()),
        "comparable_country_years": int(comparable.sum()),
        "mean_absolute_change": change.mean(),
        "maximum_absolute_change": change.max(),
        "sign_changes": int((
            np.sign(integrated.loc[comparable, column])
            != np.sign(integrated.loc[comparable, "diplomatic_score_baseline"])
        ).sum()),
        "scores_lost": scores_lost,
    })

pd.DataFrame(rows)

,diplomatic_score_baseline,diplomatic_score_zero_fill,diplomatic_score_structural_emphasis,diplomatic_score_unga_emphasis,diplomatic_score_strict_unga,diplomatic_score_exclude_provisional_participation
diplomatic_score_baseline,1.000000,0.956038,0.961519,0.959771,0.992006,0.962301
diplomatic_score_zero_fill,0.956038,1.000000,0.900600,0.936637,0.944109,0.920143
diplomatic_score_structural_emphasis,0.961519,0.900600,1.000000,0.845695,0.952910,0.940869
diplomatic_score_unga_emphasis,0.959771,0.936637,0.845695,1.000000,0.953041,0.907164
diplomatic_score_strict_unga,0.992006,0.944109,0.952910,0.953041,1.000000,0.958681
diplomatic_score_exclude_provisional_participation,0.962301,0.920143,0.940869,0.907164,0.958681,1.000000


,specification,target_country_years,comparable_country_years,mean_absolute_change,maximum_absolute_change,sign_changes,scores_lost
0,structural_emphasis,3998,3998,0.102133,0.333333,165,0
1,unga_emphasis,3998,3998,0.102133,0.333333,244,0
2,strict_unga,3997,3997,0.045193,0.750000,89,0
3,exclude_provisional_participation,149,138,0.360692,1.000000,77,11


## 5. Component relationship and disagreement diagnostics

Classify zero cases before same-sign cases to avoid mislabelling observations where both components are zero.

In [7]:
observed_components = integrated.loc[both_available].copy()
zero_case = (
    observed_components["diplomatic_structural_signed_score"].eq(0)
    | observed_components["unga_score_broad"].eq(0)
)
same_sign = (
    np.sign(observed_components["diplomatic_structural_signed_score"])
    == np.sign(observed_components["unga_score_broad"])
)
observed_components["component_sign_agreement"] = np.select(
    [zero_case, same_sign],
    ["at_least_one_zero", "same_sign"],
    default="opposite_sign",
)
observed_components["component_gap"] = (
    observed_components["diplomatic_structural_signed_score"]
    - observed_components["unga_score_broad"]
).abs()

display(pd.Series({
    "observed_component_cases": len(observed_components),
    "pearson_correlation": observed_components[[
        "diplomatic_structural_signed_score", "unga_score_broad"
    ]].corr().iloc[0, 1],
    **{
        f"agreement_{key}": int(value)
        for key, value in observed_components[
            "component_sign_agreement"
        ].value_counts().items()
    },
}, name="value"))

structural_distribution = (
    integrated.loc[
        integrated["diplomatic_structural_signed_score"].notna(),
        "diplomatic_structural_signed_score",
    ]
    .value_counts()
    .sort_index()
    .rename_axis("diplomatic_structural_signed_score")
    .reset_index(name="country_years")
)

structural_distribution["share_of_observed"] = (
    structural_distribution["country_years"]
    / structural_distribution["country_years"].sum()
)

structural_missing_summary = pd.Series(
    {
        "observed_structural_scores": int(
            integrated[
                "diplomatic_structural_signed_score"
            ].notna().sum()
        ),
        "missing_structural_scores": int(
            integrated[
                "diplomatic_structural_signed_score"
            ].isna().sum()
        ),
        "missing_share_of_full_panel": (
            integrated[
                "diplomatic_structural_signed_score"
            ].isna().mean()
        ),
    },
    name="value",
)

display(structural_missing_summary)
display(structural_distribution)

observed_component_cases       3998.000000
pearson_correlation               0.078237
agreement_at_least_one_zero    2480.000000
agreement_same_sign             924.000000
agreement_opposite_sign         594.000000
Name: value, dtype: float64

observed_structural_scores     4014.000000
missing_structural_scores       191.000000
missing_share_of_full_panel       0.045422
Name: value, dtype: float64

,diplomatic_structural_signed_score,country_years,share_of_observed
0,-1.000000,69,0.017190
1,-0.953125,13,0.003239
2,-0.875000,12,0.002990
3,-0.843750,6,0.001495
4,-0.718750,24,0.005979
5,-0.703125,1,0.000249
6,-0.625000,104,0.025909
7,-0.562500,3,0.000747
8,-0.468750,9,0.002242
9,-0.406250,2,0.000498


## 6. Distribution and selected-year diagnostics

In [8]:
display(integrated["diplomatic_score_baseline"].describe())

selected_years = [2004, 2010, 2015, 2020, 2024, 2025]
selected_year_summary = integrated.loc[
    integrated["year"].isin(selected_years)
].groupby("year").agg(
    country_years=("country_iso3", "size"),
    available_scores=("diplomatic_score_baseline", "count"),
    mean_score=("diplomatic_score_baseline", "mean"),
    median_score=("diplomatic_score_baseline", "median"),
    min_score=("diplomatic_score_baseline", "min"),
    max_score=("diplomatic_score_baseline", "max"),
)
selected_year_summary["comparability_note"] = np.where(
    selected_year_summary.index == 2025,
    "provisional_unga_only_extension",
    "complete_data_baseline_period",
)
selected_year_summary

count    4202.000000
mean       -0.277021
std         0.279138
min        -1.000000
25%        -0.436080
50%        -0.333333
75%        -0.092942
max         0.913254
Name: diplomatic_score_baseline, dtype: float64

,country_years,available_scores,mean_score,median_score,min_score,max_score,comparability_note
year,,,,,,,
2004,192,192,-0.271071,-0.340909,-0.991667,0.913254,complete_data_baseline_period
2010,191,191,-0.282248,-0.389831,-0.943776,0.845455,complete_data_baseline_period
2015,191,191,-0.235606,-0.336207,-0.952381,0.734694,complete_data_baseline_period
2020,191,191,-0.284220,-0.352273,-0.930233,0.603175,complete_data_baseline_period
2024,191,191,-0.185434,-0.250000,-1.000000,0.653333,complete_data_baseline_period
2025,191,188,-0.761076,-0.856748,-1.000000,0.678832,provisional_unga_only_extension


In [9]:
trajectory_iso3 = ["AUS", "BRA", "IND", "JPN", "PAK", "RUS", "VNM"]
integrated.loc[
    integrated["country_iso3"].isin(trajectory_iso3) & integrated["year"].isin(selected_years),
    ["country_iso3", "preferred_country_name", "year", "diplomatic_structural_signed_score",
     "unga_score_broad", "diplomatic_score_baseline", "diplomatic_integration_status"]
].sort_values(["country_iso3", "year"])

,country_iso3,preferred_country_name,year,diplomatic_structural_signed_score,unga_score_broad,diplomatic_score_baseline,diplomatic_integration_status
176,AUS,Australia,2004,0.00000,0.061538,0.030769,both_components_available
182,AUS,Australia,2010,0.00000,0.079365,0.039683,both_components_available
187,AUS,Australia,2015,0.00000,0.200000,0.100000,both_components_available
192,AUS,Australia,2020,-0.15625,0.125000,-0.015625,both_components_available
196,AUS,Australia,2024,-0.15625,0.266667,0.055208,both_components_available
197,AUS,Australia,2025,NaN,-0.565217,-0.565217,unga_only_structural_source_year_unavailable
506,BRA,Brazil,2004,0.00000,-0.787879,-0.393939,both_components_available
512,BRA,Brazil,2010,0.00000,-0.870968,-0.435484,both_components_available
517,BRA,Brazil,2015,-0.15625,-0.707692,-0.431971,both_components_available
522,BRA,Brazil,2020,0.00000,-0.443182,-0.221591,both_components_available


## 7. Final validation and provisional output schema

The prospective production panel preserves component availability and comparability explicitly. The present edition uses 2004–2024 for complete-data baseline analysis. The year 2025 remains a provisional extension and must be reported separately.

Alternative weights, the strict-UNGA score, provisional-participation exclusion, and disagreement cases belong in audit outputs rather than the principal production panel.

In [10]:
production_columns = [
    "country_iso3", "preferred_country_name", "year",
    "diplomatic_structural_signed_score",
    "structural_availability_status",
    "unga_score_broad", "unga_score_strict_yn",
    "score_status", "unga_availability_status",
    "broad_participation_rate",
    "diplomatic_integration_status",
    "diplomatic_comparability_status",
    "diplomatic_score_baseline",
]
production_candidate = integrated[production_columns].copy()

expected_integration_counts = {
    "both_components_available": 3998,
    "structural_only_unga_no_participation": 13,
    "structural_only_unga_row_unavailable": 3,
    "unga_only_structural_source_year_unavailable": 188,
    "neither_unga_no_participation_and_structural_unavailable": 2,
    "neither_unga_row_missing_and_structural_unavailable": 1,
}
actual_integration_counts = production_candidate[
    "diplomatic_integration_status"
].value_counts().to_dict()
if actual_integration_counts != expected_integration_counts:
    raise ValueError(
        "Unexpected integration-status counts: "
        f"{actual_integration_counts}"
    )

validation = pd.Series({
    "rows": len(production_candidate),
    "entities": production_candidate["country_iso3"].nunique(),
    "first_year": production_candidate["year"].min(),
    "last_year": production_candidate["year"].max(),
    "duplicate_country_years": production_candidate.duplicated(
        ["country_iso3", "year"]
    ).sum(),
    "missing_diplomatic_scores": production_candidate[
        "diplomatic_score_baseline"
    ].isna().sum(),
    "score_min": production_candidate["diplomatic_score_baseline"].min(),
    "score_max": production_candidate["diplomatic_score_baseline"].max(),
    "2025_structural_scores_present": production_candidate.loc[
        production_candidate["year"].eq(2025),
        "diplomatic_structural_signed_score",
    ].notna().sum(),
    "2025_unga_only_scores": production_candidate.loc[
        production_candidate["year"].eq(2025),
        "diplomatic_integration_status",
    ].eq("unga_only_structural_source_year_unavailable").sum(),
})

if validation["rows"] != 4205 or validation["entities"] != 192:
    raise ValueError(f"Unexpected production dimensions: {validation.to_dict()}")
if validation["duplicate_country_years"] != 0:
    raise ValueError("Duplicate production country-year keys detected.")
if validation["missing_diplomatic_scores"] != 3:
    raise ValueError("Unexpected missing diplomatic-score count.")
if validation["2025_structural_scores_present"] != 0:
    raise ValueError("2025 structural scores should remain unavailable.")
if validation["2025_unga_only_scores"] != 188:
    raise ValueError("Unexpected 2025 UNGA-only score count.")
if not production_candidate["diplomatic_score_baseline"].dropna().between(-1, 1).all():
    raise ValueError("Production diplomatic scores fall outside [-1, 1].")

scg_check = production_candidate.loc[
    production_candidate["country_iso3"].eq("SCG"),
    ["year", "diplomatic_integration_status", "diplomatic_score_baseline"],
]
if set(scg_check["year"]) != {2004, 2005, 2006}:
    raise ValueError("SCG validity window is incorrect.")
if not scg_check["diplomatic_integration_status"].eq(
    "structural_only_unga_row_unavailable"
).all():
    raise ValueError("SCG integration status is incorrect.")

tur_2025 = production_candidate.loc[
    production_candidate["country_iso3"].eq("TUR")
    & production_candidate["year"].eq(2025)
]
if len(tur_2025) != 1 or tur_2025["diplomatic_score_baseline"].notna().any():
    raise ValueError("Türkiye 2025 should remain score-unavailable.")

validation

rows                              4205.000000
entities                           192.000000
first_year                        2004.000000
last_year                         2025.000000
duplicate_country_years              0.000000
missing_diplomatic_scores            3.000000
score_min                           -1.000000
score_max                            0.913254
2025_structural_scores_present       0.000000
2025_unga_only_scores              188.000000
dtype: float64

## 8. Provisional conclusions

Use the executed diagnostics to decide whether equal conceptual weighting becomes the diplomatic-subindex baseline. Available-component renormalisation is preferred to zero filling, but 2025 remains a provisional, compositionally non-comparable UNGA-only extension until validated diplomatic-representation data become available.

Provisionally insufficient UNGA participation remains included in the baseline with an explicit exclusion sensitivity. The broad UNGA score remains the baseline and the strict Yes-versus-No score remains a robustness specification.

The prospective production output is `data/processed/diplomatic_subindex_country_year.csv`. No production file is written by this notebook.